# Agent流式输出

## 1、values输出模式
当stream_mode 设置为values模式时，每个步骤执行后，都会输出完整的状态信息，适用于每一步都要获取完整状态、状态持久化场景。

In [1]:
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
from rich import print as rprint

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = ChatOpenAI(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={'thinking':{'type':'disabled'}}
)

In [2]:
from langchain.agents import create_agent
from langchain.tools import tool
from typing import Dict, Any
from rich import print as rprint

@tool
def query_customer_data(customer_id: str) -> Dict[str, Any]:
    """
    查询客户基本信息

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户基本信息的字典，如姓名、等级、加入日期等
    """
    # 模拟数据库查询
    return {"name": "张三","level": "VIP","join_date": "2023-01-15"}


@tool
def check_order_history(customer_id: str) -> Dict[str, Any]:
    """
    查询客户订单历史

    Args:
        customer_id: 客户ID，用于唯一标识客户

    Returns:
        包含客户订单历史的字典，如总订单数、总花费等
    """
    return {"total_orders": 15,"total_spent": 25800.00}


@tool
def get_current_promotions() -> Dict[str, Any]:
    """
    获取当前可用促销活动

    Returns:
        包含当前可用促销活动的字典，如活动名称、有效日期等
    """
    return {
        "promotions": ["老用户优惠", "会员专属折扣"],
        "valid_until": "2027-01-31"
    }


# 创建客户服务Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history, get_current_promotions]
)


for chunk in customer_service_agent.stream(

        {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠"}]},
        stream_mode="values"
):
    rprint(chunk)
    print("-" * 50)

{
    'messages': [
        HumanMessage(
            content='查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='3614ab22-b278-4678-9a83-eed813faad8e'
        )
    ]
}

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='3614ab22-b278-4678-9a83-eed813faad8e'
        ),
        AIMessage(
            content='我会同时查询该客户的个人信息、订单历史和当前优惠活动。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 103,
                    'prompt_tokens': 458,
                    'total_tokens': 561,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 458
                },
                'model_provider': 'openai',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501',
                'id': '6e4a3f16-fa75-4239-9149-33fc7f4c23ad',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a14d42-69e8-7df1-ad1e-1794aa80dda5-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'CUST123456'},
                    'id': 'call_00_Qq6G15k9R5NN7mtzp2WY9707',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'CUST123456'},
                    'id': 'call_01_gPVWZWPOn6mueyHw3Lc00601',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_02_R72YLLOC7QJEJb60Gvhf5789',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 458,
                'output_tokens': 103,
                'total_tokens': 561,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        )
    ]
}

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='3614ab22-b278-4678-9a83-eed813faad8e'
        ),
        AIMessage(
            content='我会同时查询该客户的个人信息、订单历史和当前优惠活动。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 103,
                    'prompt_tokens': 458,
                    'total_tokens': 561,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 458
                },
                'model_provider': 'openai',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501',
                'id': '6e4a3f16-fa75-4239-9149-33fc7f4c23ad',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a14d42-69e8-7df1-ad1e-1794aa80dda5-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'CUST123456'},
                    'id': 'call_00_Qq6G15k9R5NN7mtzp2WY9707',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'CUST123456'},
                    'id': 'call_01_gPVWZWPOn6mueyHw3Lc00601',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_02_R72YLLOC7QJEJb60Gvhf5789',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 458,
                'output_tokens': 103,
                'total_tokens': 561,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
            name='query_customer_data',
            id='085337ea-65bd-4136-b924-6cf9f7b95a9c',
            tool_call_id='call_00_Qq6G15k9R5NN7mtzp2WY9707'
        ),
        ToolMessage(
            content='{"total_orders": 15, "total_spent": 25800.0}',
            name='check_order_history',
            id='4c2689f6-4390-4c19-992d-5914c4c9b426',
            tool_call_id='call_01_gPVWZWPOn6mueyHw3Lc00601'
        ),
        ToolMessage(
            content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
            name='get_current_promotions',
            id='5ec46dc2-9234-496e-86cb-a74ff90fc89f',
            tool_call_id='call_02_R72YLLOC7QJEJb60Gvhf5789'
        )
    ]
}

--------------------------------------------------


{
    'messages': [
        HumanMessage(
            content='查询客户ID为 CUST123456 的个人信息、历史订单和可用优惠',
            additional_kwargs={},
            response_metadata={},
            id='3614ab22-b278-4678-9a83-eed813faad8e'
        ),
        AIMessage(
            content='我会同时查询该客户的个人信息、订单历史和当前优惠活动。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 103,
                    'prompt_tokens': 458,
                    'total_tokens': 561,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 458
                },
                'model_provider': 'openai',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501',
                'id': '6e4a3f16-fa75-4239-9149-33fc7f4c23ad',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a14d42-69e8-7df1-ad1e-1794aa80dda5-0',
            tool_calls=[
                {
                    'name': 'query_customer_data',
                    'args': {'customer_id': 'CUST123456'},
                    'id': 'call_00_Qq6G15k9R5NN7mtzp2WY9707',
                    'type': 'tool_call'
                },
                {
                    'name': 'check_order_history',
                    'args': {'customer_id': 'CUST123456'},
                    'id': 'call_01_gPVWZWPOn6mueyHw3Lc00601',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_current_promotions',
                    'args': {},
                    'id': 'call_02_R72YLLOC7QJEJb60Gvhf5789',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 458,
                'output_tokens': 103,
                'total_tokens': 561,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
            name='query_customer_data',
            id='085337ea-65bd-4136-b924-6cf9f7b95a9c',
            tool_call_id='call_00_Qq6G15k9R5NN7mtzp2WY9707'
        ),
        ToolMessage(
            content='{"total_orders": 15, "total_spent": 25800.0}',
            name='check_order_history',
            id='4c2689f6-4390-4c19-992d-5914c4c9b426',
            tool_call_id='call_01_gPVWZWPOn6mueyHw3Lc00601'
        ),
        ToolMessage(
            content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
            name='get_current_promotions',
            id='5ec46dc2-9234-496e-86cb-a74ff90fc89f',
            tool_call_id='call_02_R72YLLOC7QJEJb60Gvhf5789'
        ),
        AIMessage(
            content='以下是客户 CUST123456 的查询结果：\n\n## 👤 个人信息\n| 项目 | 内容 |\n|---|---|\n| 姓名 | 
张三 |\n| 客户等级 | VIP |\n| 加入日期 | 2023-01-15 |\n\n## 📦 历史订单\n| 项目 | 数值 |\n|---|---|\n| 总订单数 | 
15 单 |\n| 总消费金额 | ¥25,800.00 |\n| 平均客单价 | 约 ¥1,720.00 |\n\n## 🎁 可用优惠\n- **老用户优惠**\n- 
**会员专属折扣**\n- 有效期至：**2027-01-31**\n\n### 💡 小结\n张三是一位加入约 3 年的 **VIP 客户**，累计消费 2.58 
万元，客单价较高。目前有「老用户优惠」和「会员专属折扣」两项优惠可用，建议优先推荐**会员专属折扣**，这与他的 VIP 
身份契合，可能带来更高转化率。如需进一步分析（如复购周期、品类偏好或计算折扣后的实际优惠力度），请告诉我。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 235,
                    'prompt_tokens': 658,
                    'total_tokens': 893,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 512},
                    'prompt_cache_hit_tokens': 512,
      

--------------------------------------------------


## 2、updates输出模式
这种模式就是默认模式。该模式中，每个步骤执行后，只增量更新状态中发生变化的内容，用于监控Agent 执行进度，例如观察Agent决定调用工具、工具执行结果等步骤。

In [3]:
for chunk in customer_service_agent.stream(
        {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},
        stream_mode="updates"
):
    rprint(chunk)
    print("-" * 50)

{
    'model': {
        'messages': [
            AIMessage(
                content='我来帮您查询客户信息和当前可用优惠。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 100,
                        'prompt_tokens': 455,
                        'total_tokens': 555,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                        'prompt_cache_hit_tokens': 256,
                        'prompt_cache_miss_tokens': 199
                    },
                    'model_provider': 'openai',
                    'model_name': 'deepseek-flash',
                    'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501',
                    'id': 'c15a1ad5-9100-4263-abdd-6618ec9b60ed',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a14d45-28de-7842-a1b7-587c5ef2ee33-0',
                tool_calls=[
                    {
                        'name': 'query_customer_data',
                        'args': {'customer_id': 'CUST123456'},
                        'id': 'call_00_DrjAXY84Isdm877wvqdm1139',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'check_order_history',
                        'args': {'customer_id': 'CUST123456'},
                        'id': 'call_01_d05qIMnEobIOZ4s2n34q8742',
                        'type': 'tool_call'
                    },
                    {
                        'name': 'get_current_promotions',
                        'args': {},
                        'id': 'call_02_YfaH5khd19HrMETYwETY6313',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 455,
                    'output_tokens': 100,
                    'total_tokens': 555,
                    'input_token_details': {'cache_read': 256},
                    'output_token_details': {}
                }
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}',
                name='query_customer_data',
                id='7b04b105-e7aa-4edb-b343-8319a43b30ba',
                tool_call_id='call_00_DrjAXY84Isdm877wvqdm1139'
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"total_orders": 15, "total_spent": 25800.0}',
                name='check_order_history',
                id='fc62a390-7cfe-4cb3-91a8-ffdf30a9f1fe',
                tool_call_id='call_01_d05qIMnEobIOZ4s2n34q8742'
            )
        ]
    }
}

--------------------------------------------------


{
    'tools': {
        'messages': [
            ToolMessage(
                content='{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}',
                name='get_current_promotions',
                id='b8cb6b62-3593-4175-a40b-9f8c9da9cdcb',
                tool_call_id='call_02_YfaH5khd19HrMETYwETY6313'
            )
        ]
    }
}

--------------------------------------------------


{
    'model': {
        'messages': [
            AIMessage(
                content='已为您查询到客户 **CUST123456** 的完整信息，整理如下：\n\n## 👤 客户基本信息\n| 项目 | 
内容 |\n|------|------|\n| 姓名 | 张三 |\n| 客户等级 | VIP |\n| 加入日期 | 2023-01-15 |\n\n## 📦 订单历史\n| 项目 |
内容 |\n|------|------|\n| 总订单数 | 15 单 |\n| 累计消费 | ¥25,800.00 |\n\n## 🎁 可用优惠\n| 项目 | 内容 
|\n|------|------|\n| 可用促销 | 老用户优惠、会员专属折扣 |\n| 有效期至 | 2027-01-31 |\n\n**小结**：张三是 VIP 
等级的资深客户（加入已超 2 年），累计消费 ¥25,800，共 15 
笔订单。目前可享受「老用户优惠」和「会员专属折扣」两项促销，均有效至 2027 年 1 月 31 
日。\n\n如需进一步了解某项优惠的具体折扣力度或适用条件，可以告诉我，我再帮您查询。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 242,
                        'prompt_tokens': 652,
                        'total_tokens': 894,
                        'completion_tokens_details': None,
                        'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 512},
                        'prompt_cache_hit_tokens': 512,
                        'prompt_cache_miss_tokens': 140
                    },
                    'model_provider': 'openai',
                    'model_name': 'deepseek-flash',
                    'system_fingerprint': 'e72d3c7f663d3e1383b00b3884cb4501',
                    'id': '45cccb6c-a9c9-48b8-a2a4-aba6f3ed00c8',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a14d45-2cc1-7e50-9c96-4cbf456001eb-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 652,
                    'output_tokens': 242,
                    'total_tokens': 894,
                    'input_token_details': {'cache_read': 512},
                    'output_token_details': {}
                }
            )
        ]
    }
}

--------------------------------------------------


## 3、 messages输出模式

该模式中会输出流式返回的Token以及相关的元数据（如：来自哪个节点），可以用在实现类似ChatGPT 的打字机效果场景，为聊天机器人等交互式应用提供最佳的实时体验。

In [6]:
for chunk in customer_service_agent.stream(
        {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},
        stream_mode="messages"
):
    #rprint(chunk)
    #print("-" * 50)

    print(chunk[0].content,end="",flush=True)

I'll query both the customer's complete information and the available promotions for you.{"promotions": ["老用户优惠", "会员专属折扣"], "valid_until": "2027-01-31"}{"name": "张三", "level": "VIP", "join_date": "2023-01-15"}{"total_orders": 15, "total_spent": 25800.0}以下是客户 **CUST123456** 的完整信息与可用优惠：

## 👤 客户基本信息
| 项目 | 内容 |
|------|------|
| 姓名 | 张三 |
| 会员等级 | VIP |
| 加入日期 | 2023-01-15 |

## 📦 订单历史
| 项目 | 内容 |
|------|------|
| 总订单数 | 15 笔 |
| 累计消费 | ¥25,800.00 |

> 平均每单约 ¥1,720，且已是 VIP 会员，属于高价值忠实客户。

## 🎁 当前可用优惠
| 项目 | 内容 |
|------|------|
| 可用促销 | 老用户优惠、会员专属折扣 |
| 有效期至 | 2027-01-31 |

**小结：** 张三是 VIP 级忠实客户（入会约 3 年、消费较高），当前可享「老用户优惠」与「会员专属折扣」两项促销，有效期至 2027-01-31，建议优先推荐会员专属折扣以提升复购。

如需进一步分析其消费趋势或核算具体优惠金额，请告诉我。

## 4、tasks模式
该模式会输出当前task任务开始和结束的时间，包含任务的结果和错误信息，该模式用于监控任务的生命周期。

In [ ]:
for chunk in customer_service_agent.stream(
        {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},
        stream_mode="tasks"
):
    rprint(chunk)
    print("-" * 50)

## 5、 debug输出模式
该模式与tasks模式类似，比task模式多输出任务步骤、时间戳、task类型（task/task_result），该模式用于调试、监控task任务的生命周期。

In [ ]:
for chunk in customer_service_agent.stream(
        {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},
        stream_mode="debug"
):
    print(chunk)
    print("-" * 50)

## 6、checkpoints输出模式
该模式中，每当检查点（checkpoint）被创建时会触发输出，输出包含检查点中的状态，用于需要状态持久化、工作流恢复或分布式执行跟踪的高级场景。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# 其他工具代码同上，保持不变
# ... ...
# 1. 创建内存检查点存储
checkpointer = InMemorySaver()

# 2. 创建Agent
customer_service_agent = create_agent(
    model=model,
    tools=[query_customer_data, check_order_history,
get_current_promotions],
    checkpointer=checkpointer  # 启用检查点
)

# 3. 创建唯一的会话ID
config = {"configurable": {"thread_id": "session01"}}

# 4. 调用Agent
checkpoint_count = 0

# 使用checkpoints模式进行流式监控
for chunk in customer_service_agent.stream(
        {"messages": [{"role": "user","content": "查询客户ID为 CUST123456 的完整信息和可用优惠"}]},
        config=config,
        stream_mode="checkpoints"
):
    checkpoint_count += 1
    print(f"检查点 #{checkpoint_count}")
    print(chunk)
    print("-" * 50)

## 7、custom输出模式
开发者通过get_stream_writer 在工具或节点内部自定义发送的数据，用于输出业务逻辑相关的进度信息（如“已处理10/100条记录”）、自定义日志或指标。

In [ ]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from langchain.tools import tool
import time

@tool
def generate_sales_report() -> str:
    """生成销售报告"""
    writer = get_stream_writer()

    writer({"type": "生成销售报告", "message": "开始生成销售报告"})

    # 模拟数据处理
    for i in range(1, 4):
        time.sleep(0.5)
        writer({"type": "生成销售报告","message": f"生成销售报告进度百分比：{i *
25}%"})

    writer({"type": "生成销售报告", "message": "报告生成完成"})

    return f"销售报告：总收入150万元，同比增长12%"


@tool
def generate_inventory_report() -> str:
    """生成库存报告"""
    writer = get_stream_writer()
    writer("开始库存分析...")
    time.sleep(0.5)
    writer("检查当前库存量...")
    time.sleep(0.5)
    writer("生成库存报告...")

    return "当前库存量为10000件，库存充足，无异常"


# 创建报告生成agent
reporting_agent = create_agent(
    model=model,
    tools=[generate_sales_report, generate_inventory_report]
)

for chunk in reporting_agent.stream(
        {"messages": [{"role": "user","content": "生成销售报告和库存报告"}]},
        stream_mode="custom"
):
    print(chunk)
    print("-" * 50)